# imports

In [14]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from statsmodels.stats.outliers_influence import variance_inflation_factor

# function defining

In [4]:
def prepare_features(df, target_col='stimmt_bioClip', verbose=True):
    """
    Split a dataframe into features/target and preprocess features
    for statsmodels (imputation, scaling, one-hot encoding, intercept).

    Parameters
    ----------
    df : pd.DataFrame
        Input data containing features and the target column.
    target_col : str
        Name of the binary target column.
    verbose : bool
        Whether to print a summary of the data.

    Returns
    -------
    X_df : pd.DataFrame
        Preprocessed feature matrix with constant column added.
    y : pd.Series
        Target as integers.
    preprocessor : ColumnTransformer
        Fitted preprocessor (useful for transforming new data).
    """
    # --- 1. Split features / target ---
    X = df.drop(columns=[target_col])
    y = df[target_col].astype(int)

    # --- 2. Identify column types ---
    categorical_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    numerical_cols   = X.select_dtypes(include=['number']).columns.tolist()

    if verbose:
        print(f"Sample size:        {len(X):,}")
        print(f"Outcome rate:       {y.mean():.3f} ({y.sum():,} positives, {(1-y).sum():,} negatives)")
        print(f"Numerical features:   {len(numerical_cols)}")
        print(f"Categorical features: {len(categorical_cols)}")
        print("-" * 60)

    # --- 3. Preprocessing pipelines ---
    numerical_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
    ])
    categorical_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot',  OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False)),
    ])
    preprocessor = ColumnTransformer(transformers=[
        ('num', numerical_pipeline,   numerical_cols),
        ('cat', categorical_pipeline, categorical_cols),
    ])

    # --- 4. Transform features ---
    X_transformed = preprocessor.fit_transform(X)

    # Reconstruct feature names
    ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
    encoded_cat_names = ohe.get_feature_names_out(categorical_cols).tolist()
    feature_names = numerical_cols + encoded_cat_names

    X_df = pd.DataFrame(X_transformed, columns=feature_names, index=X.index)
    X_df = sm.add_constant(X_df)

    return X_df, y, preprocessor, numerical_cols

In [5]:
def check_multicollinearity(df, numerical_cols, X_df, corr_threshold=0.7, verbose=True):
    """
    Check multicollinearity via pairwise correlations among numerical
    features and VIF on the full design matrix.

    Parameters
    ----------
    df : pd.DataFrame
        Original (untransformed) data, used for the correlation check.
    numerical_cols : list of str
        Names of numerical feature columns in df.
    X_df : pd.DataFrame
        Preprocessed design matrix (including constant) for VIF.
    corr_threshold : float
        Absolute correlation above which pairs are flagged.
    verbose : bool
        Whether to print results.

    Returns
    -------
    high_corr : pd.Series
        Flagged feature pairs and their correlations.
    vif : pd.DataFrame
        VIF for each column of X_df, sorted descending.
    """
    # --- Correlation among numerical features ---
    num_corr = df[numerical_cols].corr()
    high_corr = num_corr.where(np.abs(num_corr) > corr_threshold).stack().dropna()
    high_corr = high_corr[
        high_corr.index.get_level_values(0) < high_corr.index.get_level_values(1)
        ]

    # --- VIF on the full design matrix ---
    vif = pd.DataFrame({
        "feature": X_df.columns,
        "VIF": [variance_inflation_factor(X_df.values, i) for i in range(X_df.shape[1])]
    }).sort_values("VIF", ascending=False)

    if verbose:
        print(f"Strong correlations (|r| > {corr_threshold}):")
        print(high_corr if len(high_corr) else "  (none)")
        print("-" * 60)
        print(vif.to_string(index=False))

    return high_corr, vif

In [6]:
def fit_logistic_regression(X_df, y, save_path=None, verbose=True):
    """
    Fit a logistic regression with statsmodels and build a coefficient
    summary table with CIs, odds ratios, and p-values.

    Parameters
    ----------
    X_df : pd.DataFrame
        Preprocessed design matrix (including constant).
    y : pd.Series
        Binary target.
    save_path : str or None
        If given, the coefficient table is saved to this CSV path.
    verbose : bool
        Whether to print model summary and coefficients.

    Returns
    -------
    result : statsmodels results object
        The fitted model (for predictions, diagnostics, etc.).
    summary_sorted : pd.DataFrame
        Coefficient table sorted by |coef|, intercept first.
    """
    # --- 5. Fit ---
    if verbose:
        print("Fitting logistic regression...")
    logit_model = sm.Logit(y, X_df)
    result = logit_model.fit(disp=False, maxiter=1000)

    # --- 6. Model-level summary ---
    if verbose:
        print("-" * 60)
        print(f"Pseudo R² (McFadden): {result.prsquared:.4f}")
        print(f"Log-likelihood:        {result.llf:.1f}")
        print(f"LLR p-value:           {result.llr_pvalue:.2e}")
        print(f"Converged:             {result.mle_retvals.get('converged', 'n/a')}")
        print("-" * 60)

    # --- 7. Coefficient table ---
    coefs = result.params
    conf = result.conf_int()
    conf.columns = ["ci_lower", "ci_upper"]
    summary = pd.DataFrame({
        "coef":        coefs,
        "std_err":     result.bse,
        "z":           result.tvalues,
        "p_value":     result.pvalues,
        "odds_ratio":  np.exp(coefs),
        "or_ci_lower": np.exp(conf["ci_lower"]),
        "or_ci_upper": np.exp(conf["ci_upper"]),
    })

    intercept_row = summary.loc[["const"]] if "const" in summary.index else summary.iloc[0:0]
    other_rows = summary.drop("const", errors="ignore")
    other_rows = other_rows.reindex(other_rows["coef"].abs().sort_values(ascending=False).index)
    summary_sorted = pd.concat([intercept_row, other_rows])

    if verbose:
        with pd.option_context("display.float_format", lambda x: f"{x:.4f}"):
            print("\nCoefficients (sorted by |coef|, intercept first):")
            print(summary_sorted)

    # --- 8. Save ---
    if save_path is not None:
        summary_sorted.to_csv(save_path)
        if verbose:
            print(f"\n✓ Saved to {save_path}")

    return result, summary_sorted

In [7]:
def run_logit_analysis(df, target_col, save_path=None,
                       corr_threshold=0.7, verbose=True):
    """Full pipeline: preprocess → multicollinearity check → logistic regression."""
    X_df, y, preprocessor, numerical_cols = prepare_features(df, target_col, verbose)
    high_corr, vif = check_multicollinearity(df, numerical_cols, X_df, corr_threshold, verbose)
    result, coef_table = fit_logistic_regression(X_df, y, save_path, verbose)
    return {
        "result": result,
        "coef_table": coef_table,
        "vif": vif,
        "high_corr": high_corr,
        "preprocessor": preprocessor,
        "X_df": X_df,
        "y": y,
    }

# loading data

In [10]:
path_all = '/Users/krolik/Documents/python_/MScThesis/diopsis_apollo_all/md/final_python/bioclip_top1_rank_order_meta.csv'

df= pd.read_csv(path_all)

In [17]:
lg_clean_1 = df[['width_px','blur_score','altitude','shadow','blur',
                            'site','day_night_twilight','rank_order_stimmt'
                            ]]

lg_clean_2_1 = df[['width_px', 'blur_score', 'altitude', 'shadow', 'blur', 'site',
       'day_night_twilight', 'rank_order_stimmt','colorfulness',
       'brightness_mean', 'dynamic_range', 'underexposed_pct',
       'overexposed_pct', 'edge_density',
       'saturation_mean', 'distinct_hues'
]]

lg_clean_3_1 = df[['width_px', 'blur_score', 'altitude', 'shadow', 'blur', 'site',
       'day_night_twilight', 'rank_order_stimmt','colorfulness', 'top_1_rank_score',
       'brightness_mean', 'dynamic_range', 'underexposed_pct',
       'overexposed_pct', 'edge_density',
       'saturation_mean', 'distinct_hues'
]]

# applying functions

In [16]:
log_clean1 = run_logit_analysis(df=lg_clean_1, target_col='rank_order_stimmt',save_path='/Users/krolik/Downloads/Thesis/log1.csv')

/var/folders/qf/bfjql8w96nzdw1vj0z7dbzzr0000gn/T/ipykernel_39695/1026916212.py:29: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()


Sample size:        39,788
Outcome rate:       0.278 (11,055 positives, 28,733 negatives)
Numerical features:   5
Categorical features: 2
------------------------------------------------------------
Strong correlations (|r| > 0.7):
  (none)
------------------------------------------------------------
                    feature      VIF
                      const 3.805164
              site_Monstein 1.443144
   day_night_twilight_night 1.426186
         site_Weissfluhjoch 1.382296
                   altitude 1.294412
                   width_px 1.106575
                     shadow 1.103720
                 blur_score 1.082716
                       blur 1.045811
day_night_twilight_twilight 1.007130
Fitting logistic regression...
------------------------------------------------------------
Pseudo R² (McFadden): 0.0859
Log-likelihood:        -21492.5
LLR p-value:           0.00e+00
Converged:             True
------------------------------------------------------------

Coefficients (so

In [18]:
log_clean2_1 = run_logit_analysis(df=lg_clean_2_1, target_col='rank_order_stimmt',save_path='/Users/krolik/Downloads/Thesis/log2_1.csv')

/var/folders/qf/bfjql8w96nzdw1vj0z7dbzzr0000gn/T/ipykernel_39695/1026916212.py:29: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()


Sample size:        39,788
Outcome rate:       0.278 (11,055 positives, 28,733 negatives)
Numerical features:   13
Categorical features: 2
------------------------------------------------------------
Strong correlations (|r| > 0.7):
  (none)
------------------------------------------------------------
                    feature      VIF
               colorfulness 7.017207
              dynamic_range 5.016211
            saturation_mean 4.946715
                      const 4.611227
              distinct_hues 2.961279
               edge_density 2.888825
            brightness_mean 2.728516
                 blur_score 2.147079
         site_Weissfluhjoch 1.717510
              site_Monstein 1.714468
           underexposed_pct 1.673649
   day_night_twilight_night 1.582741
            overexposed_pct 1.489875
                   altitude 1.337534
                   width_px 1.283585
                     shadow 1.240807
                       blur 1.189522
day_night_twilight_twilight 1.0

In [19]:
log_clean3_1 = run_logit_analysis(df=lg_clean_3_1, target_col='rank_order_stimmt',save_path='/Users/krolik/Downloads/Thesis/log3_1.csv')

/var/folders/qf/bfjql8w96nzdw1vj0z7dbzzr0000gn/T/ipykernel_39695/1026916212.py:29: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()


Sample size:        39,788
Outcome rate:       0.278 (11,055 positives, 28,733 negatives)
Numerical features:   14
Categorical features: 2
------------------------------------------------------------
Strong correlations (|r| > 0.7):
  (none)
------------------------------------------------------------
                    feature      VIF
               colorfulness 7.190509
            saturation_mean 5.042242
              dynamic_range 5.016484
                      const 4.615902
              distinct_hues 2.961382
               edge_density 2.957807
            brightness_mean 2.732525
                 blur_score 2.148608
         site_Weissfluhjoch 1.717676
              site_Monstein 1.714729
           underexposed_pct 1.674218
   day_night_twilight_night 1.585421
           top_1_rank_score 1.505117
            overexposed_pct 1.502577
                   width_px 1.383441
                   altitude 1.337784
                     shadow 1.251433
                       blur 1.1